Previous notebook showed that the CATE estimates I was getting were basically 0. 

In this notebook I've going to redo the same analysis but aplly corrections to the basefile that help focus the causal questions that we have asked.

Most notably redefining the basefile structure to be area hour rather than station hour: The formal statement is that the potential outcome for unit i depends only on unit i's own treatment, not on the treatment assigned to any other unit. Written precisely: Y_i(t_1, ..., t_n) = Y_i(t_i) for all possible treatment vectors. Violation of this is what Sobel (2006) and others call spillover or interference, and it is essentially guaranteed in your setting.
The causal mechanism is modal substitution. A strike on, say, the Central Line means thousands of commuters who would normally travel from Holborn to Liverpool Street underground are now standing on the street looking for alternatives. Some take buses, some walk, some use Santander Bikes. The demand shock at a bike station near Holborn is not caused by that station's own strike_exposed value — it is caused partly by the aggregated treatment status of all nearby stations and all nearby tube lines. Station A's potential outcome under control Y_A(0) is not the same whether station B next to it is treated or not. Your model assumes it is.
This creates two concrete problems for your CATE estimates. First, your control group is contaminated. On a strike day, stations coded as strike_exposed = 0 — those not directly adjacent to a striking line — may still see elevated ridership because displaced commuters spread out across a wider area. If those control observations have inflated Y values, the counterfactual comparison is biased downward, which would attenuate your estimated treatment effect toward zero. This is consistent with the S-Learner's near-zero ATE. Second, your standard errors are underestimated. The asymptotic normality proofs underlying CausalForestDML (Wager & Athey, 2018) assume independent potential outcomes. Spatial clustering of treatment effects violates this, so your confidence intervals will be too narrow. (from Claude)

And also to redefine strike_effected as not a binary treatment but continous, based on the number of tubes striking.

Will also look at stratified sampling to remove superflous control rows

In [2]:
import pandas as pd
import h3
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

from econml.metalearners import SLearner, TLearner, XLearner
from econml.dml import CausalForestDML
from xgboost import XGBRegressor, XGBClassifier

e:\tfl_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#bring in data
PATH = "bike_hourly_with_covariates_v2_filtered.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'is_school_holiday', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
       'wind_speed_10m', 'weather_code', 'dist_nearest_tube_km',
       'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score'],
      dtype='str')

In [8]:
def assign_h3_cell(lat, lon, resolution=8):
    # Resolution 8 gives ~460m edge hexagons, roughly a 10-min walk radius
    return h3.latlng_to_cell(lat, lon, resolution)

df["h3_cell"] = df.apply(lambda r: assign_h3_cell(r["lat"], r["lon"]), axis=1)

In [11]:
df[["h3_cell"]].head()

,h3_cell
0,88194ad347fffff
1,88194ad347fffff
2,88194ad347fffff
3,88194ad347fffff
4,88194ad347fffff


In [12]:
df.columns

Index(['station_id', 'trips_start', 'date', 'ts', 'y_log1p', 'strike_exposed',
       'lat', 'lon', 'station_name', 'weather_cell_id', 'hour', 'dow', 'month',
       'year', 'doy', 'is_weekend', 'is_am_peak', 'is_pm_peak',
       'is_bank_holiday', 'is_school_holiday', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover',
       'wind_speed_10m', 'weather_code', 'dist_nearest_tube_km',
       'n_tube_within_500m', 'n_tube_within_1km', 'strike_severity_daily_frac',
       'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score',
       'h3_cell'],
      dtype='str')

In [14]:

# Aggregate to cell-hour level
cell_hour = (
    df.groupby(["h3_cell", "trips_start"])
    .agg(
        total_trips     = ("ts", "sum"),
        frac_exposed    = ("strike_exposed", "mean"),  # continuous treatment
        n_stations      = ("station_id", "nunique"),

        # Keep covariates by averaging within cell
        temperature_2m  = ("temperature_2m", "mean"),
        relative_humidity_2m = ("relative_humidity_2m", "mean"),
        precipitation   = ("precipitation", "mean"),
        rain            = ("rain", "mean"),
        cloud_cover     = ("cloud_cover", "mean"),
        wind_speed_10m = ("wind_speed_10m", "mean"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "mean"),
        hour = ("hour", "first"),  # hour of day is constant within cell-hour
        is_am_peak      = ("is_am_peak", "first"),
        is_pm_peak      = ("is_pm_peak", "first"),
        is_school_holiday = ("is_school_holiday", "first"),
        is_bank_holiday = ("is_bank_holiday", "first"),
        dist_nearest_tube_km = ("dist_nearest_tube_km", "mean"),
        days_since_last_strike = ("days_since_last_strike", "mean"),
        cycle_infra_score = ("cycle_infra_score", "mean"),
        days_to_next_strike = ("days_to_next_strike", "mean"),
        doy = ("doy", "first"),
        is_weekend = ("is_weekend", "first"),

        # ... other covariates
    )
    .reset_index()
)


cell_hour["y_log1p"] = np.log1p(cell_hour["total_trips"])
# Treatment is now continuous: fraction of stations in the cell that are strike-exposed
# This also addresses the hidden versions of treatment problem (see Consistency below)

In [15]:
# You cannot directly observe supply constraints from trips_start alone,
# but you can proxy for them using the ratio of bikes taken to time window.
# A station that records its maximum-ever hourly trips is likely supply-constrained.

# Compute per-station maximum historical hourly trips (from your original data)
# before aggregating to cell level
station_max_hourly = (
    df.groupby("station_id")["ts"]
    .max()
    .rename("station_max_hourly_trips")
    .reset_index()
)

df = df.merge(station_max_hourly, on="station_id", how="left")

# Flag observations where trips are near the historical maximum
# (likely supply-constrained)
df["near_capacity"] = (df["ts"] >= df["station_max_hourly_trips"] * 0.9).astype(int)

# Aggregate to cell level — fraction of stations near capacity
# If this is high on strike days, supply constraint is your problem
cell_capacity = (
    df.groupby(["h3_cell", "trips_start"])
    .agg(frac_near_capacity=("near_capacity", "mean"))
    .reset_index()
)

cell_hour = cell_hour.merge(cell_capacity, on=["h3_cell", "trips_start"], how="left")

# Check: are strike-exposed cells more likely to be near capacity?
print(cell_hour.groupby("frac_exposed")["frac_near_capacity"].mean())

frac_exposed
0.000000    0.000190
0.100000    0.000000
0.111111    0.000000
0.125000    0.000000
0.142857    0.000000
0.166667    0.001208
0.181818    0.000000
0.200000    0.000000
0.222222    0.000000
0.250000    0.000265
0.272727    0.000000
0.285714    0.001091
0.300000    0.000000
0.333333    0.000421
0.375000    0.000000
0.400000    0.000000
0.428571    0.000000
0.444444    0.000000
0.500000    0.000698
0.555556    0.000000
0.571429    0.000000
0.600000    0.000000
0.625000    0.000000
0.666667    0.000000
0.700000    0.002273
0.714286    0.000000
0.750000    0.001147
0.777778    0.004630
0.800000    0.000000
0.818182    0.000000
0.833333    0.000728
0.857143    0.000000
0.875000    0.005208
0.888889    0.000000
0.900000    0.000000
0.909091    0.001818
0.916667    0.000000
1.000000    0.000866
Name: frac_near_capacity, dtype: float64


In [16]:
# Step 2: Create the missing outcome columns
# Adjust the column names to match whatever your aggregation produced

# If your aggregation output column is called "total_trips":
cell_hour["y_log1p"] = np.log1p(cell_hour["total_trips"])
cell_hour["y_per_station_log1p"] = np.log1p(
    cell_hour["total_trips"] / cell_hour["n_stations"]
)

# If it's called something else (e.g. "ts" or "trips_start"):
# cell_hour["y_log1p"] = np.log1p(cell_hour["ts"])
# cell_hour["y_per_station_log1p"] = np.log1p(cell_hour["ts"] / cell_hour["n_stations"])

# Quick sanity check before re-running diagnostics
print(cell_hour[["total_trips", "n_stations", "y_log1p", "y_per_station_log1p"]].describe())
print("\nAny NaNs:", cell_hour[["y_log1p", "y_per_station_log1p"]].isna().sum().to_dict())

        total_trips    n_stations       y_log1p  y_per_station_log1p
count  2.487918e+06  2.487918e+06  2.487918e+06         2.487918e+06
mean   1.047163e+01  3.152981e+00  1.939356e+00         1.198403e+00
std    1.598175e+01  2.216496e+00  9.478790e-01         4.719305e-01
min    1.000000e+00  1.000000e+00  6.931472e-01         6.931472e-01
25%    2.000000e+00  1.000000e+00  1.098612e+00         6.931472e-01
50%    5.000000e+00  3.000000e+00  1.791759e+00         1.098612e+00
75%    1.300000e+01  4.000000e+00  2.639057e+00         1.455287e+00
max    4.280000e+02  1.200000e+01  6.061457e+00         4.779123e+00

Any NaNs: {'y_log1p': 0, 'y_per_station_log1p': 0}


In [17]:
# Priority 1: Is the naive difference already negative in raw data?
treated_mean = cell_hour.loc[cell_hour["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_hour.loc[cell_hour["frac_exposed"] == 0, "y_log1p"].mean()
print(f"Naive difference: {treated_mean - control_mean:.4f}")


# Priority 2: Is n_stations confounding the outcome?
print(cell_hour.groupby(
    pd.cut(cell_hour["frac_exposed"], bins=[-0.01, 0, 0.5, 1.01],
           labels=["control", "partial", "fully_treated"])
)[["n_stations", "y_log1p", "y_per_station_log1p"]].mean())



Naive difference: 0.2765
               n_stations   y_log1p  y_per_station_log1p
frac_exposed                                            
control          3.147291  1.937254             1.197811
partial          4.872033  2.601125             1.374301
fully_treated    3.644144  2.113956             1.250326


In [18]:
cell_hour.shape

(2487918, 26)

In [19]:
cell_hour.head()

,h3_cell,trips_start,total_trips,frac_exposed,n_stations,temperature_2m,relative_humidity_2m,precipitation,rain,cloud_cover,...,is_bank_holiday,dist_nearest_tube_km,days_since_last_strike,cycle_infra_score,days_to_next_strike,doy,is_weekend,y_log1p,frac_near_capacity,y_per_station_log1p
0,88194ad101fffff,2016-01-10 02:00:00,1,0.0,1,6.3,85.0,0.0,0.0,81.0,...,0,1.598242,30.0,65.0,30.0,10,1,0.693147,0.0,0.693147
1,88194ad101fffff,2016-01-10 06:00:00,2,0.0,2,5.4,80.0,0.0,0.0,100.0,...,0,1.577844,30.0,63.0,30.0,10,1,1.098612,0.0,0.693147
2,88194ad101fffff,2016-01-10 08:00:00,2,0.0,1,5.7,83.0,0.0,0.0,99.0,...,0,1.557445,30.0,61.0,30.0,10,1,1.098612,0.0,1.098612
3,88194ad101fffff,2016-01-10 09:00:00,1,0.0,1,5.9,84.0,0.0,0.0,98.0,...,0,1.557445,30.0,61.0,30.0,10,1,0.693147,0.0,0.693147
4,88194ad101fffff,2016-01-10 10:00:00,2,0.0,2,6.3,83.0,0.0,0.0,78.0,...,0,1.645024,30.0,59.0,30.0,10,1,1.098612,0.0,0.693147


In [20]:
COL_TIME = "trips_start"
COL_Y    = "total_trips"
COL_T    = "frac_exposed"

cell_hour[COL_TIME] = pd.to_datetime(cell_hour[COL_TIME])
cell_hour[COL_Y] = pd.to_numeric(cell_hour[COL_Y], errors="coerce")
cell_hour[COL_T] = pd.to_numeric(cell_hour[COL_T], errors="coerce").fillna(0).astype(int)

cell_hour = cell_hour.dropna(subset=[COL_TIME, COL_Y])
cell_hour["y_log1p"] = np.log1p(cell_hour[COL_Y].astype(float))

cell_hour[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,trips_start,total_trips,frac_exposed,y_log1p
0,2016-01-10 02:00:00,1,0,0.693147
1,2016-01-10 06:00:00,2,0,1.098612
2,2016-01-10 08:00:00,2,0,1.098612
3,2016-01-10 09:00:00,1,0,0.693147
4,2016-01-10 10:00:00,2,0,1.098612


In [21]:
SPLIT_DATE = "2018-01-01"

train = cell_hour[cell_hour[COL_TIME] < SPLIT_DATE].copy()
test  = cell_hour[cell_hour[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (1646656, 26) test: (841262, 26)


In [22]:
# Candidate numeric features (only keep those that exist)
num_cols = [
    "hour", 
    "dow",
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_am_peak",
    "is_pm_peak",
    "is_bank_holiday",
    "lat",
    "lon",
    'is_school_holiday',
    'dist_nearest_tube_km',
    'n_tube_within_500m', 
    'n_tube_within_1km', 
    'strike_severity_daily_frac',
    'days_to_next_strike',
    'days_since_last_strike',
    'cycle_infra_score',
    'n_stations'
]


#cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

#print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

num_cols: ['hour', 'doy', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_weekend', 'is_am_peak', 'is_pm_peak', 'is_bank_holiday', 'is_school_holiday', 'dist_nearest_tube_km', 'strike_severity_daily_frac', 'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score', 'n_stations']
Xtr: (1646656, 19) Xte: (841262, 19)


In [23]:
print("Treatment rate:", T_train.mean())
print("Treated count:", T_train.sum(), "of", len(T_train))

Treatment rate: 0.0047429457032920055
Treated count: 7810 of 1646656


In [24]:
rf_y = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

rf_t = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=0,
)

In [25]:
xg_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

xg_t = XGBClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)


Treated: 7,810  |  Control: 1,638,846  |  scale_pos_weight: 209.8


In [21]:
import time

start = time.time()

s = SLearner(overall_model=xg_y)
s.fit(Y_train, T_train, X=Xtr)
tau_s_learner = s.effect(Xte)

s_learner_end = time.time() - start

print("SLearner done in", s_learner_end / 60, "minutes")

tlearner_start = time.time()

t = TLearner(models=xg_y
)
t.fit(Y_train, T_train, X=Xtr)
tau_t_learner = t.effect(Xte)

t_learner_end = time.time() - tlearner_start

print("TLearner done in", t_learner_end / 60, "minutes")

Xlearner_start = time.time()

x = XLearner(models=xg_y,
             propensity_model=xg_t)

x.fit(Y_train, T_train, X=Xtr)
tau_x_learner = x.effect(Xte)

xlearner_end = time.time() - Xlearner_start

print("XLearner done in", xlearner_end / 60, "minutes")

# Map each learner name to its corresponding tau array.
tau_map = {
    "s_learner": tau_s_learner,
    "t_learner": tau_t_learner,
    "x_learner": tau_x_learner,
}

att_mask = T_test > 0

for model_name, tau_values in tau_map.items():
    ate = np.mean(tau_values)
    att = np.mean(tau_values[att_mask])
    atc = np.mean(tau_values[~att_mask])

    print(f"{model_name.upper()} ATE: {np.expm1(ate)*100:.2f}%")
    print(f"{model_name.upper()} ATT: {np.expm1(att)*100:.2f}%  (effect on actually treated cells)")
    print(f"{model_name.upper()} ATC: {np.expm1(atc)*100:.2f}%  (effect if controls were treated)")


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


SLearner done in 1.2105539679527282 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TLearner done in 1.0827234427134196 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XLearner done in 2.965574737389882 minutes
S_LEARNER ATE: 0.65%
S_LEARNER ATT: 0.45%  (effect on actually treated cells)
S_LEARNER ATC: 0.65%  (effect if controls were treated)
T_LEARNER ATE: -8.54%
T_LEARNER ATT: -7.36%  (effect on actually treated cells)
T_LEARNER ATC: -8.55%  (effect if controls were treated)
X_LEARNER ATE: -3.80%
X_LEARNER ATT: -4.98%  (effect on actually treated cells)
X_LEARNER ATC: -3.79%  (effect if controls were treated)


In [22]:
# Assuming your cell-hour dataframe is called cell_hour
# and has columns: frac_exposed (or strike_exposed), y_log1p, n_stations, h3_cell, trips_start

# 1. Basic treated vs control comparison
treated_mean = cell_hour.loc[cell_hour["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_hour.loc[cell_hour["frac_exposed"] == 0, "y_log1p"].mean()
naive_diff   = treated_mean - control_mean

print(f"Mean Y (treated cells) : {treated_mean:.4f}")
print(f"Mean Y (control cells) : {control_mean:.4f}")
print(f"Naive difference       : {naive_diff:.4f}")

# 2. Check n_stations distribution across treated vs control
print("\nStations per cell (treated vs control):")
print(cell_hour.groupby("frac_exposed")["n_stations"].describe())

# 3. Check how many cells are ever treated
ever_treated = cell_hour.groupby("h3_cell")["frac_exposed"].max()
print(f"\nCells ever treated  : {(ever_treated > 0).sum()}")
print(f"Cells never treated : {(ever_treated == 0).sum()}")
print(f"Fraction treated    : {(ever_treated > 0).mean():.3f}")

# 4. Check the outcome distribution
print("\nY distribution:")
print(cell_hour.groupby("frac_exposed")["y_log1p"].describe())

# 5. Time of day — are you including night hours in treatment?
print("\nTreatment rate by hour:")# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

# ── Nuisance model for Y (outcome) ───────────────────────────────────────────
# Standard regression — no imbalance issue here.
# n_estimators deliberately modest; the causal forest adds further complexity.
xgb_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Nuisance model for T (propensity) ────────────────────────────────────────
# Binary target (0/1) fit as regression — outputs E[T|X] = P(T=1|X).
# scale_pos_weight is the key addition vs. Random Forest.
xgb_t = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── CausalForestDML ───────────────────────────────────────────────────────────
cf_xgb = CausalForestDML(
    model_y          = xgb_y,
    model_t          = xgb_t,
    n_estimators     = 200,       # more trees → smoother CATE surface
    min_samples_leaf = 50,
    max_features     = "auto",
    n_jobs           = -1,
    random_state     = 0,
)

cf_xgb.fit(Y_train, T_train, X=Xtr)
tau_cf_xgb   = cf_xgb.effect(Xte)
lb_xgb, ub_xgb = cf_xgb.effect_interval(Xte, alpha=0.05)

print(f"\nATE (XGB nuisance): {np.mean(tau_cf_xgb):.6f}")
print(f"95% CI:            [{np.mean(lb_xgb):.6f}, {np.mean(ub_xgb):.6f}]")
print(cell_hour.groupby("hour")["frac_exposed"].mean().round(4))

Mean Y (treated cells) : 1.9693
Mean Y (control cells) : 1.9392
Naive difference       : 0.0301

Stations per cell (treated vs control):
                  count      mean       std  min  25%  50%  75%   max
frac_exposed                                                         
0             2475438.0  3.152608  2.215590  1.0  1.0  3.0  4.0  12.0
1               12480.0  3.227003  2.388467  1.0  1.0  3.0  4.0  12.0

Cells ever treated  : 128
Cells never treated : 4
Fraction treated    : 0.970

Y distribution:
                  count      mean       std       min       25%       50%  \
frac_exposed                                                                
0             2475438.0  1.939205  0.947411  0.693147  1.098612  1.791759   
1               12480.0  1.969262  1.036224  0.693147  1.098612  1.791759   

                   75%       max  
frac_exposed                      
0             2.639057  6.061457  
1             2.708050  5.880533  

Treatment rate by hour:
Treated: 7,81

In [23]:
# How many cell-hours are actually treated at cell level?
n_treated_cells = (T_train > 0).sum()
n_total         = len(T_train)
treatment_rate  = n_treated_cells / n_total

print(f"Treatment rate  : {treatment_rate:.4%}")
print(f"Treated cells   : {n_treated_cells:,}")
print(f"Control cells   : {n_total - n_treated_cells:,}")
print(f"Ratio           : 1 treated per {int(1/treatment_rate):,} rows")

# Also check min_samples_leaf relative to treated count
min_samples_leaf = 50
max_leaves_treated = n_treated_cells // min_samples_leaf
print(f"\nWith min_samples_leaf={min_samples_leaf}:")
print(f"T-Learner treated model can have at most {max_leaves_treated} leaves")
print(f"to partition {n_treated_cells:,} treated observations")

Treatment rate  : 0.4743%
Treated cells   : 7,810
Control cells   : 1,638,846
Ratio           : 1 treated per 210 rows

With min_samples_leaf=50:
T-Learner treated model can have at most 156 leaves
to partition 7,810 treated observations


In [26]:
#let's save this dataframe for later use in meta-learners notebook
cell_hour.to_parquet("cell_hour_for_meta_learners.parquet", index=False)

In [45]:
def stratified_subsample(
    X: np.ndarray,
    Y: np.ndarray,
    T: np.ndarray,
    control_ratio: float = 10.0,
    random_state: int = 42,
) -> tuple:
    """
    Keep all treated units. Subsample controls to control_ratio × n_treated.
    Returns X, Y, T, sample_weights arrays of equal length.

    sample_weights correct for the induced sampling distribution:
      - treated units weight = 1.0
      - sampled control units weight = n_control_total / n_control_sampled
    
    Passing these weights to CausalForestDML.fit(..., sample_weight=w)
    ensures the propensity model E[T|X] remains calibrated to the true
    population treatment rate, not the artificially inflated sample rate.
    """
    rng = np.random.default_rng(random_state)

    # For continuous treatment — any non-zero exposure is "treated"
    treated_idx      = np.where(T > 0)[0]
    control_idx      = np.where(T == 0)[0]

    n_treated        = len(treated_idx)
    n_control_total  = len(control_idx)
    n_control_sample = min(int(n_treated * control_ratio), n_control_total)

    sampled_control_idx = rng.choice(control_idx, size=n_control_sample, replace=False)
    keep_idx = np.concatenate([treated_idx, sampled_control_idx])
    rng.shuffle(keep_idx)

    control_weight = n_control_total / n_control_sample
    weights = np.where(T[keep_idx] > 0, 1.0, control_weight)

    print(f"Treated kept   : {n_treated:>10,}")
    print(f"Controls kept  : {n_control_sample:>10,}  (of {n_control_total:,})")
    print(f"Control weight : {control_weight:.2f}x")
    print(f"Total train    : {len(keep_idx):>10,}  (was {len(T):,})")
    print(f"Sample treat %  : {T[keep_idx][T[keep_idx]>0].shape[0]/len(keep_idx)*100:.2f}%")
    print(f"True treat %    : {(T>0).mean()*100:.2f}%")

    return X[keep_idx], Y[keep_idx], T[keep_idx], weights

In [1]:
Xtr_sub, Y_sub, T_sub, sample_weights = stratified_subsample(
    Xtr, Y_train, T_train,
    control_ratio=10.0,
    random_state=42,
)

NameError: name 'stratified_subsample' is not defined

In [ ]:
Xtr_sub.shape

(85910, 17)

In [ ]:
start = time.time()

s = SLearner(overall_model=xg_y)
s.fit(Y_sub, T_sub, X=Xtr_sub)
tau_s_learner = s.effect(Xte)

s_learner_end = time.time() - start

print("SLearner done in", s_learner_end / 60, "minutes")

tlearner_start = time.time()

t = TLearner(models=xg_y
)
t.fit(Y_sub, T_sub, X=Xtr_sub)
tau_t_learner = t.effect(Xte)

t_learner_end = time.time() - tlearner_start

print("TLearner done in", t_learner_end / 60, "minutes")

Xlearner_start = time.time()

x = XLearner(models=xg_y,
             propensity_model=xg_t)

x.fit(Y_sub, T_sub, X=Xtr_sub)
tau_x_learner = x.effect(Xte)

xlearner_end = time.time() - Xlearner_start

print("XLearner done in", xlearner_end / 60, "minutes")

# Map each learner name to its corresponding tau array.
tau_map = {
    "s_learner": tau_s_learner,
    "t_learner": tau_t_learner,
    "x_learner": tau_x_learner,
}

att_mask = T_test > 0

for model_name, tau_values in tau_map.items():
    ate = np.mean(tau_values)
    att = np.mean(tau_values[att_mask])
    atc = np.mean(tau_values[~att_mask])

    print(f"{model_name.upper()} ATE: {np.expm1(ate)*100:.2f}%")
    print(f"{model_name.upper()} ATT: {np.expm1(att)*100:.2f}%  (effect on actually treated cells)")
    print(f"{model_name.upper()} ATC: {np.expm1(atc)*100:.2f}%  (effect if controls were treated)")

In [ ]:
cf_xgb.fit(Y_sub, T_sub, X=Xtr_sub, sample_weight=sample_weights)
tau_cf_xgb   = cf_xgb.effect(Xte)
lb_xgb, ub_xgb = cf_xgb.effect_interval(Xte, alpha=0.05)

print(f"\nATE (XGB nuisance): {np.mean(tau_cf_xgb):.6f}")
print(f"95% CI:            [{np.mean(lb_xgb):.6f}, {np.mean(ub_xgb):.6f}]")
print(cell_hour.groupby("hour")["strike_exposed"].mean().round(4))

NameError: name 'cf_xgb' is not defined

In [ ]:
# Quantify how much the treatment sparsity is costing you
# Relative efficiency compared to a balanced experiment
e_x         = T_train.mean()
efficiency  = e_x * (1 - e_x) / 0.25   # 0.25 is the maximum (balanced at 50/50)
n_effective = len(T_train) * efficiency

print(f"True propensity       : {e_x:.4f}")
print(f"Relative efficiency   : {efficiency:.4f}")
print(f"Effective sample size : {n_effective:,.0f}  (of {len(T_train):,} actual rows)")
print(f"Equivalent balanced N : {n_effective:,.0f} rows at 50/50 treatment")

Let's bring back in the cell hour dataframe, and aggregate up to cell day

In [27]:
#bring in data
PATH = "cell_hour_for_meta_learners.parquet"  # adjust

df = pd.read_parquet(PATH)

df.size
df.columns

Index(['h3_cell', 'trips_start', 'total_trips', 'frac_exposed', 'n_stations',
       'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'cloud_cover', 'wind_speed_10m', 'strike_severity_daily_frac', 'hour',
       'is_am_peak', 'is_pm_peak', 'is_school_holiday', 'is_bank_holiday',
       'dist_nearest_tube_km', 'days_since_last_strike', 'cycle_infra_score',
       'days_to_next_strike', 'doy', 'is_weekend', 'y_log1p',
       'frac_near_capacity', 'y_per_station_log1p'],
      dtype='str')

In [37]:
df['day'] = pd.to_datetime(df["trips_start"]).dt.date

# Aggregate cell-hour to cell-day
cell_day = (
    df.groupby(["h3_cell", "day"])
    .agg(
        total_trips          = ("total_trips", "sum"),
        n_stations           = ("n_stations", "first"),
        frac_exposed         = ("frac_exposed", "max"),  # any exposure = treated
        temperature_2m       = ("temperature_2m", "mean"),
        relative_humidity_2m = ("relative_humidity_2m", "mean"),
        precipitation        = ("precipitation", "mean"),
        rain                 = ("rain", "mean"),
        doy                  = ("doy", "first"),
     #   dow                 = ("dow", "first"),
        cloud_cover          = ("cloud_cover", "mean"),
        wind_speed_10m      = ("wind_speed_10m", "mean"),
        strike_severity_daily_frac = ("strike_severity_daily_frac", "mean"),
        is_weekend           = ("is_weekend", "first"),
        is_bank_holiday      = ("is_bank_holiday", "first"),
        is_school_holiday    = ("is_school_holiday", "first"),
        dist_nearest_tube_km = ("dist_nearest_tube_km", "first"),
        days_to_next_strike  = ("days_to_next_strike", "first"),
        days_since_last_strike = ("days_since_last_strike", "first"),
        cycle_infra_score    = ("cycle_infra_score", "first")
    )
    .reset_index()
)

cell_day["y_per_station_log1p"] = np.log1p(
    cell_day["total_trips"] / cell_day["n_stations"]
)

print(f"Cell-day treatment rate: {(cell_day['frac_exposed'] > 0).mean():.4%}")
print(f"Treated cell-days      : {(cell_day['frac_exposed'] > 0).sum():,}")

Cell-day treatment rate: 0.7095%
Treated cell-days      : 1,008


In [38]:
cell_day.head()

,h3_cell,day,total_trips,n_stations,frac_exposed,temperature_2m,relative_humidity_2m,precipitation,rain,doy,...,wind_speed_10m,strike_severity_daily_frac,is_weekend,is_bank_holiday,is_school_holiday,dist_nearest_tube_km,days_to_next_strike,days_since_last_strike,cycle_infra_score,y_per_station_log1p
0,88194ad101fffff,2016-01-10,42,1,0,6.392308,80.307692,0.000000,0.000000,10,...,20.076923,0.0,1,0,0,1.598242,30.0,30.0,65.000000,3.761200
1,88194ad101fffff,2016-01-11,46,2,0,5.753846,91.538462,0.169231,0.169231,11,...,13.746154,0.0,0,0,0,1.577844,30.0,30.0,63.000000,3.178054
2,88194ad101fffff,2016-01-12,39,3,0,4.790909,80.363636,0.036364,0.036364,12,...,24.909091,0.0,0,0,0,1.615831,30.0,30.0,59.666667,2.639057
3,88194ad101fffff,2016-01-13,58,1,0,3.573333,81.933333,0.000000,0.000000,13,...,15.620000,0.0,0,0,0,1.598242,30.0,30.0,65.000000,4.077537
4,88194ad101fffff,2016-01-14,59,1,0,2.535294,76.647059,0.029412,0.029412,14,...,24.070588,0.0,0,0,0,1.598242,30.0,30.0,65.000000,4.094345


In [39]:
COL_TIME = "day"
COL_Y    = "total_trips"
COL_T    = "frac_exposed"

cell_day[COL_TIME] = pd.to_datetime(cell_day[COL_TIME])
cell_day[COL_Y] = pd.to_numeric(cell_day[COL_Y], errors="coerce")
cell_day[COL_T] = pd.to_numeric(cell_day[COL_T], errors="coerce").fillna(0).astype(int)

cell_day = cell_day.dropna(subset=[COL_TIME, COL_Y])
cell_day["y_log1p"] = np.log1p(cell_day[COL_Y].astype(float))

cell_day[[COL_TIME, COL_Y, COL_T, "y_log1p"]].head()

,day,total_trips,frac_exposed,y_log1p
0,2016-01-10,42,0,3.761200
1,2016-01-11,46,0,3.850148
2,2016-01-12,39,0,3.688879
3,2016-01-13,58,0,4.077537
4,2016-01-14,59,0,4.094345


In [40]:
SPLIT_DATE = "2018-01-01"

train = cell_day[cell_day[COL_TIME] < SPLIT_DATE].copy()
test  = cell_day[cell_day[COL_TIME] >= SPLIT_DATE].copy()

print("train:", train.shape, "test:", test.shape)

Y_train = train["y_log1p"].values
T_train = train[COL_T].values.astype(int)

Y_test = test["y_log1p"].values
T_test = test[COL_T].values.astype(int)

train: (93909, 22) test: (48162, 22)


In [41]:
# Candidate numeric features (only keep those that exist)
num_cols = [
    "month",
    "year",
    "doy",
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "cloud_cover",
    "wind_speed_10m",
    "is_weekend",
    "is_bank_holiday",
    'is_school_holiday',
    'dist_nearest_tube_km',
    'n_tube_within_500m', 
    'n_tube_within_1km', 
    'strike_severity_daily_frac',
    'days_to_next_strike',
    'days_since_last_strike',
    'cycle_infra_score',
    'n_stations'
]


#cat_cols = [c for c in cat_cols if c in train.columns]
num_cols = [c for c in num_cols if c in train.columns]

#print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

pre = ColumnTransformer(
    transformers=[
      #  ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3
)

Xtr = pre.fit_transform(train[num_cols])
Xte = pre.transform(test[num_cols])

print("Xtr:", Xtr.shape, "Xte:", Xte.shape)

num_cols: ['doy', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain', 'cloud_cover', 'wind_speed_10m', 'is_weekend', 'is_bank_holiday', 'is_school_holiday', 'dist_nearest_tube_km', 'strike_severity_daily_frac', 'days_to_next_strike', 'days_since_last_strike', 'cycle_infra_score', 'n_stations']
Xtr: (93909, 16) Xte: (48162, 16)


In [42]:
xg_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

xg_t = XGBClassifier(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

Treated: 607  |  Control: 93,302  |  scale_pos_weight: 153.7


In [43]:
import time

start = time.time()

s = SLearner(overall_model=xg_y)
s.fit(Y_train, T_train, X=Xtr)
tau_s_learner = s.effect(Xte)

s_learner_end = time.time() - start

print("SLearner done in", round(s_learner_end / 60, 2), "minutes")

tlearner_start = time.time()

t = TLearner(models=xg_y
)
t.fit(Y_train, T_train, X=Xtr)
tau_t_learner = t.effect(Xte)

t_learner_end = time.time() - tlearner_start

print("TLearner done in", round(t_learner_end / 60, 2), "minutes")

Xlearner_start = time.time()

x = XLearner(models=xg_y,
             propensity_model=xg_t)

x.fit(Y_train, T_train, X=Xtr)
tau_x_learner = x.effect(Xte)

xlearner_end = time.time() - Xlearner_start

print("XLearner done in", round(xlearner_end / 60, 2), "minutes")

# Map each learner name to its corresponding tau array.
tau_map = {
    "s_learner": tau_s_learner,
    "t_learner": tau_t_learner,
    "x_learner": tau_x_learner,
}

att_mask = T_test > 0

for model_name, tau_values in tau_map.items():
    ate = np.mean(tau_values)
    att = np.mean(tau_values[att_mask])
    atc = np.mean(tau_values[~att_mask])

    print(f"{model_name.upper()} ATE: {np.expm1(ate)*100:.2f}%")
    print(f"{model_name.upper()} ATT: {np.expm1(att)*100:.2f}%  (effect on actually treated cells)")
    print(f"{model_name.upper()} ATC: {np.expm1(atc)*100:.2f}%  (effect if controls were treated)")

e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


SLearner done in 0.07 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


TLearner done in 0.05 minutes


e:\tfl_project\.venv\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


XLearner done in 0.13 minutes
S_LEARNER ATE: 1.62%
S_LEARNER ATT: 0.60%  (effect on actually treated cells)
S_LEARNER ATC: 1.63%  (effect if controls were treated)
T_LEARNER ATE: 0.29%
T_LEARNER ATT: -25.50%  (effect on actually treated cells)
T_LEARNER ATC: 0.54%  (effect if controls were treated)
X_LEARNER ATE: 1.48%
X_LEARNER ATT: -3.77%  (effect on actually treated cells)
X_LEARNER ATC: 1.52%  (effect if controls were treated)


In [44]:
# Assuming your cell-hour dataframe is called cell_hour
# and has columns: frac_exposed (or strike_exposed), y_log1p, n_stations, h3_cell, trips_start

# 1. Basic treated vs control comparison
treated_mean = cell_day.loc[cell_day["frac_exposed"] > 0, "y_log1p"].mean()
control_mean = cell_day.loc[cell_day["frac_exposed"] == 0, "y_log1p"].mean()
naive_diff   = treated_mean - control_mean

print(f"Mean Y (treated cells) : {treated_mean:.4f}")
print(f"Mean Y (control cells) : {control_mean:.4f}")
print(f"Naive difference       : {naive_diff:.4f}")

# 2. Check n_stations distribution across treated vs control
print("\nStations per cell (treated vs control):")
print(cell_day.groupby("frac_exposed")["n_stations"].describe())

# 3. Check how many cells are ever treated
ever_treated = cell_day.groupby("h3_cell")["frac_exposed"].max()
print(f"\nCells ever treated  : {(ever_treated > 0).sum()}")
print(f"Cells never treated : {(ever_treated == 0).sum()}")
print(f"Fraction treated    : {(ever_treated > 0).mean():.3f}")

# 4. Check the outcome distribution
print("\nY distribution:")
print(cell_day.groupby("frac_exposed")["y_log1p"].describe())

# 5. Time of day — are you including night hours in treatment?
print("\nTreatment rate by hour:")# ── Compute scale_pos_weight for the propensity model ────────────────────────
# This is the standard XGBoost recommendation for imbalanced binary targets:
# ratio of negative to positive cases.
n_control = (T_train == 0).sum()
n_treated = (T_train == 1).sum()
spw       = n_control / n_treated

print(f"Treated: {n_treated:,}  |  Control: {n_control:,}  |  scale_pos_weight: {spw:.1f}")

# ── Nuisance model for Y (outcome) ───────────────────────────────────────────
# Standard regression — no imbalance issue here.
# n_estimators deliberately modest; the causal forest adds further complexity.
xgb_y = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 6,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 50,      # analogous to min_samples_leaf — keeps leaves stable
    tree_method       = "hist",  # fast histogram method, works well on large data
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── Nuisance model for T (propensity) ────────────────────────────────────────
# Binary target (0/1) fit as regression — outputs E[T|X] = P(T=1|X).
# scale_pos_weight is the key addition vs. Random Forest.
xgb_t = XGBRegressor(
    n_estimators      = 500,
    learning_rate     = 0.05,
    max_depth         = 5,        # shallower — propensity is a simpler surface than Y
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    min_child_weight  = 20,       # lower than Y model — treated group is small
    scale_pos_weight  = spw,      # key imbalance correction
    tree_method       = "hist",
    n_jobs            = -1,
    random_state      = 0,
    verbosity         = 0,
)

# ── CausalForestDML ───────────────────────────────────────────────────────────
cf_xgb = CausalForestDML(
    model_y          = xgb_y,
    model_t          = xgb_t,
    n_estimators     = 200,       # more trees → smoother CATE surface
    min_samples_leaf = 50,
    max_features     = "auto",
    n_jobs           = -1,
    random_state     = 0,
)

cf_xgb.fit(Y_train, T_train, X=Xtr)
tau_cf_xgb   = cf_xgb.effect(Xte)
lb_xgb, ub_xgb = cf_xgb.effect_interval(Xte, alpha=0.05)

print(f"\nATE (XGB nuisance): {np.mean(tau_cf_xgb):.6f}")
print(f"95% CI:            [{np.mean(lb_xgb):.6f}, {np.mean(ub_xgb):.6f}]")
print(cell_hour.groupby("hour")["frac_exposed"].mean().round(4))

Mean Y (treated cells) : 5.0076
Mean Y (control cells) : 4.6945
Naive difference       : 0.3131

Stations per cell (treated vs control):
                 count      mean       std  min  25%  50%  75%   max
frac_exposed                                                        
0             141063.0  1.584696  1.024718  1.0  1.0  1.0  2.0  10.0
1               1008.0  1.603175  1.017850  1.0  1.0  1.0  2.0   9.0

Cells ever treated  : 128
Cells never treated : 4
Fraction treated    : 0.970

Y distribution:
                 count      mean       std       min       25%       50%  \
frac_exposed                                                               
0             141063.0  4.694493  1.152716  0.693147  4.043051  4.859812   
1               1008.0  5.007628  1.146819  0.693147  4.391344  5.105945   

                   75%       max  
frac_exposed                      
0             5.509388  7.391415  
1             5.927592  7.380256  

Treatment rate by hour:
Treated: 607  |  Cont

In [51]:
cell_day["day"]

0        2016-01-10
1        2016-01-11
2        2016-01-12
3        2016-01-13
4        2016-01-14
            ...    
142066   2018-12-28
142067   2018-12-29
142068   2018-12-30
142069   2018-12-31
142070   2019-01-01
Name: day, Length: 142071, dtype: datetime64[s]

In [52]:
import statsmodels.formula.api as smf
import numpy as np

# Ensure you have a proper date column at day level
cell_day["date_str"] = cell_day["day"].astype(str)
cell_day["treated"]  = (cell_day["frac_exposed"] > 0).astype(int)

# ── Model 1: Basic two-way fixed effects ─────────────────────────────────────
# Cell FE absorbs all time-invariant differences between cells
# Date FE absorbs all shocks common to all cells on a given day
twfe = smf.ols(
    "y_per_station_log1p ~ treated + C(h3_cell) + C(date_str)",
    data=cell_day
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cell_day["h3_cell"]}
    # Cluster at cell level: within-cell observations across time
    # are correlated, so row-level SEs would be anti-conservative
)

ate_twfe = np.expm1(twfe.params["treated"]) * 100
ci_lo    = np.expm1(twfe.conf_int().loc["treated", 0]) * 100
ci_hi    = np.expm1(twfe.conf_int().loc["treated", 1]) * 100

print(f"TWFE ATE  : {ate_twfe:+.2f}%")
print(f"95% CI    : [{ci_lo:.2f}%, {ci_hi:.2f}%]")
print(f"p-value   : {twfe.pvalues['treated']:.4f}")
print(f"N         : {twfe.nobs:,.0f}")
print(f"R²        : {twfe.rsquared:.4f}")

# ── Model 2: Add weather and calendar controls ────────────────────────────────
# TWFE with covariates is more efficient — residual variance is lower
# which tightens the confidence interval
twfe_controls = smf.ols(
    """y_per_station_log1p ~ treated 
       + temperature_2m 
       + precipitation
       + is_weekend
       + is_bank_holiday
       + is_school_holiday
       + days_to_next_strike
       + days_since_last_strike
       + C(h3_cell) 
       + C(date_str)""",
    data=cell_day
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cell_day["h3_cell"]}
)

ate_ctrl = np.expm1(twfe_controls.params["treated"]) * 100
ci_lo_c  = np.expm1(twfe_controls.conf_int().loc["treated", 0]) * 100
ci_hi_c  = np.expm1(twfe_controls.conf_int().loc["treated", 1]) * 100

print(f"\nTWFE + controls ATE : {ate_ctrl:+.2f}%")
print(f"95% CI              : [{ci_lo_c:.2f}%, {ci_hi_c:.2f}%]")
print(f"p-value             : {twfe_controls.pvalues['treated']:.4f}")
print(f"R²                  : {twfe_controls.rsquared:.4f}")
# R² should be substantially higher than the basic TWFE
# because weather and calendar explain a lot of residual variance

TWFE ATE  : +10.27%
95% CI    : [4.62%, 16.22%]
p-value   : 0.0003
N         : 142,071
R²        : 0.7877


ValueError: The weights and list don't have the same length.

In [53]:
# ── Define the control variables ──────────────────────────────────────────────
control_vars = [
    "y_per_station_log1p",
    "treated",
    "temperature_2m",
    "precipitation",
    "is_weekend",
    "is_bank_holiday",
    "is_school_holiday",
    "days_to_next_strike",
    "days_since_last_strike",
    "h3_cell",
    "date_str",
]

# Drop NaNs only from the columns the model will actually use
# This ensures the groups array and the model data have identical length
cell_day_clean = cell_day[control_vars].dropna().copy()

print(f"Rows before dropna : {len(cell_day):,}")
print(f"Rows after dropna  : {len(cell_day_clean):,}")
print(f"Rows dropped       : {len(cell_day) - len(cell_day_clean):,}")

# Check which columns are causing the drops
print("\nNaN counts per column:")
print(cell_day[control_vars].isna().sum())

Rows before dropna : 142,071
Rows after dropna  : 141,939
Rows dropped       : 132

NaN counts per column:
y_per_station_log1p         0
treated                     0
temperature_2m            132
precipitation             132
is_weekend                  0
is_bank_holiday             0
is_school_holiday           0
days_to_next_strike       132
days_since_last_strike    132
h3_cell                     0
date_str                    0
dtype: int64


In [54]:
twfe_controls = smf.ols(
    """y_per_station_log1p ~ treated 
       + temperature_2m 
       + precipitation
       + is_weekend
       + is_bank_holiday
       + is_school_holiday
       + days_to_next_strike
       + days_since_last_strike
       + C(h3_cell) 
       + C(date_str)""",
    data=cell_day_clean
).fit(
    cov_type="cluster",
    cov_kwds={"groups": cell_day_clean["h3_cell"]}  # <-- uses clean subset, not cell_day
)

ate_ctrl = np.expm1(twfe_controls.params["treated"]) * 100
ci_lo_c  = np.expm1(twfe_controls.conf_int().loc["treated", 0]) * 100
ci_hi_c  = np.expm1(twfe_controls.conf_int().loc["treated", 1]) * 100

print(f"\nTWFE + controls ATE : {ate_ctrl:+.2f}%")
print(f"95% CI              : [{ci_lo_c:.2f}%, {ci_hi_c:.2f}%]")
print(f"p-value             : {twfe_controls.pvalues['treated']:.4f}")
print(f"R²                  : {twfe_controls.rsquared:.4f}")


TWFE + controls ATE : +10.27%
95% CI              : [4.59%, 16.25%]
p-value             : 0.0003
R²                  : 0.7887
